# Insurance Enrollment Prediction & Agentic Outreach Assistant

## Introduction

The objective of this project is to build an end-to-end machine learning solution that predicts whether an employee is likely to enroll in a benefits program. The project goes beyond model building by focusing on real-world data challenges such as missing values, duplicate records, inconsistent text formats, invalid dates, and integrating information from multiple datasets.

The workflow includes data cleaning, feature engineering, exploratory analysis, model training, honest evaluation, and fairness considerations. Special attention is given to identifying and removing leaky features, handling sentinel values correctly, and ensuring that demographic information is not used inappropriately for decision-making.

In addition to the predictive model, this project implements an Agentic Outreach Assistant that helps HR teams prioritize employees for outreach based on predicted enrollment probability and regional outreach capacity. The assistant can make predictions, explain model outputs, retrieve regional information, rank outreach candidates, and explicitly refuse requests that attempt to use forbidden or leaky features.

## Notebook Outline

1. Multi-Table Data Investigation & Cleaning
2. Feature Engineering
3. Model Training & Honest Evaluation
4. Agentic Outreach Assistant
5. Conclusion

## Business Problem

The HR Benefits team cannot contact every employee during the enrollment period because each region has a limited outreach capacity.

The objective is to predict which employees are most likely to enroll and prioritize outreach while avoiding target leakage and ensuring fair, explainable decision-making.

# 1. Multi-Table Data Investigation & Cleaning

In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# read dataset
employees = pd.read_csv("employees_raw.csv")
regions = pd.read_csv("region_benefit_profiles.csv")

In [3]:
# profile both datasets
# employee dataset
employees.head()

,employee_id,age,gender,marital_status,salary,employment_type,region,has_dependents,tenure_years,enrolled,application_date,last_contact_date,last_contact_channel,plan_tier_requested,broker_channel,prior_year_enrolled,legacy_propensity_score,outreach_notes
0,12324,28,Male,Divorced,44047.60,Full-time,West,No,1.4,0,2024-04-14,2024-03-30,SMS,basic,Employer-Sponsored,0,0.104,Spouse covered elsewhere
1,17825,23,Male,Married,80111.28,Full-time,Midwest,Yes,0.5,1,2024-05-09,2024-04-18,Call,STANDARD,Direct,-1,0.874,Requested more time
2,15200,39,Male,Married,69855.65,Full-time,South,Yes,3.4,1,22/06/2024,2024-06-09,none,Silver,Direct,0,0.870,No response to first outreach
3,16690,31,Female,Married,91567.33,Full-time,South,No,1.2,1,2024-08-16,2024-07-29,email,premium plan,Employer-Sponsored,1,0.894,Requested more time
4,17465,42,Female,Single,59861.68,Contract,Midwest,Yes,23.8,0,13/05/2024,2024-05-06,email,Standard,Employer-Sponsored,-1,0.163,Requested more time


In [4]:
employees.shape

(10008, 18)

In [5]:
# data types
employees.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10008 entries, 0 to 10007
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   employee_id              10008 non-null  int64  
 1   age                      10008 non-null  int64  
 2   gender                   10008 non-null  object 
 3   marital_status           10008 non-null  object 
 4   salary                   10008 non-null  float64
 5   employment_type          10008 non-null  object 
 6   region                   10008 non-null  object 
 7   has_dependents           10008 non-null  object 
 8   tenure_years             10008 non-null  float64
 9   enrolled                 10008 non-null  int64  
 10  application_date         9289 non-null   object 
 11  last_contact_date        10008 non-null  object 
 12  last_contact_channel     8823 non-null   object 
 13  plan_tier_requested      9482 non-null   object 
 14  broker_channel        

In [6]:
# missing values
employees.isnull().sum()

employee_id                   0
age                           0
gender                        0
marital_status                0
salary                        0
employment_type               0
region                        0
has_dependents                0
tenure_years                  0
enrolled                      0
application_date            719
last_contact_date             0
last_contact_channel       1185
plan_tier_requested         526
broker_channel              470
prior_year_enrolled           0
legacy_propensity_score     908
outreach_notes             1453
dtype: int64

In [7]:
# duplicates
employees["employee_id"].duplicated().sum()

np.int64(8)

In [8]:
# overall statiscs
employees.describe()

,employee_id,age,salary,tenure_years,enrolled,prior_year_enrolled,legacy_propensity_score
count,10008.000000,10008.000000,10008.000000,10008.000000,10008.000000,10008.000000,9100.000000
mean,14999.756994,43.003197,65034.633918,3.966447,0.617106,-0.047062,0.605136
std,2886.750141,12.284623,14921.509635,3.894377,0.486117,0.871637,0.363105
min,10001.000000,22.000000,2207.790000,0.000000,0.000000,-1.000000,0.010000
25%,12500.750000,33.000000,54717.165000,1.200000,0.000000,-1.000000,0.191000
50%,14998.500000,43.000000,65063.065000,2.800000,1.000000,0.000000,0.814000
75%,17499.250000,54.000000,75053.687500,5.600000,1.000000,1.000000,0.907000
max,20000.000000,64.000000,120312.000000,36.000000,1.000000,1.000000,0.990000


In [9]:
# categorical data
object_cols = employees.select_dtypes(include="object").columns

for col in object_cols:
    print(employees[col].value_counts())
    print()

gender
Male      4820
Female    4813
Other      375
Name: count, dtype: int64

marital_status
Married     4595
Single      3879
Divorced    1001
Widowed      533
Name: count, dtype: int64

employment_type
Full-time    7046
Part-time    1975
Contract      987
Name: count, dtype: int64

region
West         2586
Northeast    2508
Midwest      2488
South        2426
Name: count, dtype: int64

has_dependents
Yes    5999
No     4009
Name: count, dtype: int64

application_date
2024-04-08     36
2024-12-19     33
2024-01-25     32
2024-11-03     30
2024-09-17     30
               ..
21/06/2024      1
10-Feb-2024     1
17-Jun-2024     1
09-Mar-2024     1
04-Feb-2024     1
Name: count, Length: 1049, dtype: int64

last_contact_date
2024-10-19    43
2024-11-02    41
2024-01-22    40
2024-09-24    39
2024-10-27    39
              ..
2024-12-27     4
2024-12-24     3
2023-12-04     3
2023-12-03     2
2023-12-06     2
Name: count, Length: 390, dtype: int64

last_contact_channel
EMAIL     1166
email

In [10]:
# unique
employees.nunique()

employee_id                10000
age                           43
gender                         3
marital_status                 4
salary                      9988
employment_type                3
region                         4
has_dependents                 2
tenure_years                 241
enrolled                       2
application_date            1049
last_contact_date            390
last_contact_channel          12
plan_tier_requested           15
broker_channel                 3
prior_year_enrolled            3
legacy_propensity_score      663
outreach_notes                 6
dtype: int64

In [11]:
# region dataset
regions.head()

,region,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level
0,Midwest,2488,0.617,64921.46,595,4.4,469,18,High
1,Northeast,2506,0.612,65008.32,455,3.1,151,38,low
2,South,2424,0.628,65200.49,481,3.3,324,17,MED
3,West,2582,0.613,65007.07,574,4.7,437,28,Low


In [12]:
regions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   region                       4 non-null      object 
 1   n_employees_region           4 non-null      int64  
 2   hist_enrollment_rate_region  4 non-null      float64
 3   avg_salary_region            4 non-null      float64
 4   avg_premium_cost_usd         4 non-null      int64  
 5   benefits_broker_rating       4 non-null      float64
 6   hr_outreach_capacity         4 non-null      int64  
 7   open_enrollment_window_days  4 non-null      int64  
 8   state_mandate_level          4 non-null      object 
dtypes: float64(3), int64(4), object(2)
memory usage: 420.0+ bytes


In [13]:
regions.describe(include="all")

,region,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level
count,4,4.000000,4.000000,4.000000,4.000000,4.0000,4.00000,4.000000,4
unique,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
top,Midwest,NaN,NaN,NaN,NaN,NaN,NaN,NaN,High
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
mean,NaN,2500.000000,0.617500,65034.335000,526.250000,3.8750,345.25000,25.250000,NaN
std,NaN,65.012819,0.007326,117.994929,68.631261,0.7932,143.66251,9.844626,NaN
min,NaN,2424.000000,0.612000,64921.460000,455.000000,3.1000,151.00000,17.000000,NaN
25%,NaN,2472.000000,0.612750,64985.667500,474.500000,3.2500,280.75000,17.750000,NaN
50%,NaN,2497.000000,0.615000,65007.695000,527.500000,3.8500,380.50000,23.000000,NaN
75%,NaN,2525.000000,0.619750,65056.362500,579.250000,4.4750,445.00000,30.500000,NaN


In [14]:
regions.isnull().sum()

region                         0
n_employees_region             0
hist_enrollment_rate_region    0
avg_salary_region              0
avg_premium_cost_usd           0
benefits_broker_rating         0
hr_outreach_capacity           0
open_enrollment_window_days    0
state_mandate_level            0
dtype: int64

In [15]:
regions.nunique()

region                         4
n_employees_region             4
hist_enrollment_rate_region    4
avg_salary_region              4
avg_premium_cost_usd           4
benefits_broker_rating         4
hr_outreach_capacity           4
open_enrollment_window_days    4
state_mandate_level            4
dtype: int64

In [16]:
regions.duplicated().sum()

np.int64(0)

Data Cleaning

In [17]:
df = employees.copy()

In [18]:
# Parse the messy operational fields
# Application Date
df["application_date"] = pd.to_datetime( df["application_date"], errors="coerce" )

In [19]:
# last_contact_date
df["last_contact_date"] = pd.to_datetime( df["last_contact_date"], errors="coerce" )

In [20]:
# impossible order
invalid_dates = df[ df["last_contact_date"] > df["application_date"]]

invalid_dates

,employee_id,age,gender,marital_status,salary,employment_type,region,has_dependents,tenure_years,enrolled,application_date,last_contact_date,last_contact_channel,plan_tier_requested,broker_channel,prior_year_enrolled,legacy_propensity_score,outreach_notes
59,18661,49,Male,Married,38545.55,Full-time,Northeast,No,0.0,0,2024-03-23,2024-03-28,none,Basic,Employer-Sponsored,-1,0.017,Requested more time
513,18287,26,Male,Married,66528.59,Full-time,Northeast,Yes,0.4,1,2024-03-06,2024-03-08,email,Standard,Third-Party,-1,0.897,Requested more time
882,13294,41,Female,Married,48157.91,Full-time,Northeast,Yes,9.2,1,2024-03-11,2024-03-15,email,Basic,NaN,1,0.898,Attended benefits webinar
1120,12726,55,Female,Single,38144.55,Part-time,Midwest,Yes,10.2,0,2024-03-16,2024-03-24,Text,BASIC,Employer-Sponsored,0,0.135,Requested more time
1162,13675,45,Female,Single,48729.11,Full-time,South,Yes,0.5,1,2024-05-20,2024-05-27,EMAIL,Bronze,Third-Party,-1,0.763,Declined - cost concern
1219,16644,58,Other,Divorced,82597.28,Full-time,Midwest,No,0.2,1,2024-04-21,2024-04-28,NaN,standard,Third-Party,-1,0.883,Spouse covered elsewhere
1226,10064,31,Male,Divorced,53910.54,Part-time,West,Yes,1.1,0,2024-01-16,2024-01-19,e-mail,Bronze,NaN,1,0.230,Spouse covered elsewhere
1552,18876,45,Male,Married,70082.53,Part-time,Midwest,Yes,2.2,1,2024-09-29,2024-10-08,e-mail,Standard,Employer-Sponsored,1,0.976,Follow-up scheduled
1615,13503,34,Female,Divorced,56946.16,Full-time,Northeast,Yes,0.1,1,2024-01-28,2024-02-06,Email,Silver,Direct,-1,0.961,Declined - cost concern
1625,19665,48,Female,Single,67456.03,Full-time,Northeast,Yes,13.7,1,2024-12-07,2024-12-09,Email,standard,Direct,1,0.810,Declined - cost concern


In [21]:
# clean these dates
df.loc[ df["last_contact_date"] > df["application_date"], "last_contact_date"] = pd.NaT

Contact dates occurring after the application date were treated as invalid operational records. These dates were replaced with missing values (NaT) to prevent future information from influencing model predictions.

In [22]:
# plan_tier_requested
df["plan_tier_requested"] = (df["plan_tier_requested"].str.lower().str.strip())

# coverted to lowercase and removed spaces 

In [23]:
df["plan_tier_requested"].value_counts()

plan_tier_requested
standard        3925
basic           1512
silver          1306
silver plan     1265
bronze           532
premium          309
gold             290
premium plan     201
gold plan        142
Name: count, dtype: int64

In [24]:
df["plan_tier_requested"] = (df["plan_tier_requested"].replace({
        "premium plan": "premium",
        "gold plan": "gold",
        "silver plan": "silver",
        "basic plan": "basic"
    })
)

In [25]:
 #prior_year_enrolled
df["prior_year_enrolled"].value_counts()

prior_year_enrolled
-1    4048
 1    3577
 0    2383
Name: count, dtype: int64

Handle the special sentinel value in 'prior_year_enrolled'. Convert numeric values into meaningful categories:
-1 → new_hire (no previous enrollment record)
 0 → not_enrolled
 1 → enrolled
 This preserves the business meaning and prevents the model from treating the sentinel as a normal binary value.

In [26]:
df["last_contact_channel"].value_counts(dropna=False)

last_contact_channel
NaN       1185
EMAIL     1166
email     1130
e-mail    1109
Email     1100
PHONE      624
Phone      619
phone      602
Call       587
Text       500
sms        490
SMS        488
none       408
Name: count, dtype: int64

In [27]:
df["last_contact_channel"] = (df["last_contact_channel"].str.lower().str.strip())

In [28]:
df["last_contact_channel"] = df["last_contact_channel"].replace({
    "e-mail": "email",
    "mail": "email",
    "phone call": "phone",
    "telephone": "phone",
    "text": "sms"
})

In [29]:
# duplicates
df["employee_id"].duplicated().sum()

np.int64(8)

In [30]:
df[df["employee_id"].duplicated(keep=False)][["employee_id", "enrolled", "last_contact_date"]].sort_values("employee_id")


,employee_id,enrolled,last_contact_date
6582,10311,1,2024-03-29
9835,10311,0,2024-03-29
455,10484,0,2024-11-06
4188,10484,1,2024-11-06
1327,13441,0,2024-05-29
9665,13441,1,2024-05-29
6139,13840,0,2024-08-07
7606,13840,1,2024-08-07
1589,14846,0,2024-12-19
7370,14846,1,2024-12-19


In [31]:
# there is no unique combinatoon
# so keep the latest ones only
df = df.sort_values("last_contact_date")

df = df.drop_duplicates(subset="employee_id",keep="last")

In [32]:
df.duplicated().sum()

np.int64(0)

In [33]:
region_df = regions.copy()

In [34]:
region_df.head()

,region,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level
0,Midwest,2488,0.617,64921.46,595,4.4,469,18,High
1,Northeast,2506,0.612,65008.32,455,3.1,151,38,low
2,South,2424,0.628,65200.49,481,3.3,324,17,MED
3,West,2582,0.613,65007.07,574,4.7,437,28,Low


In [35]:
region_df.isnull().sum()
# no null 

region                         0
n_employees_region             0
hist_enrollment_rate_region    0
avg_salary_region              0
avg_premium_cost_usd           0
benefits_broker_rating         0
hr_outreach_capacity           0
open_enrollment_window_days    0
state_mandate_level            0
dtype: int64

In [36]:
region_df.duplicated().sum()
# no duplicates 

np.int64(0)

In [37]:
# inspect categorical columns
region_df["state_mandate_level"].value_counts(dropna=False)

state_mandate_level
High    1
low     1
MED     1
Low     1
Name: count, dtype: int64

In [38]:
# format to lowercase and remove blanks
region_df["state_mandate_level"] = (region_df["state_mandate_level"].str.lower().str.strip())

In [39]:
region_df["state_mandate_level"].value_counts()

state_mandate_level
low     2
high    1
med     1
Name: count, dtype: int64

In [40]:
region_df.describe()

,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days
count,4.000000,4.000000,4.000000,4.000000,4.0000,4.00000,4.000000
mean,2500.000000,0.617500,65034.335000,526.250000,3.8750,345.25000,25.250000
std,65.012819,0.007326,117.994929,68.631261,0.7932,143.66251,9.844626
min,2424.000000,0.612000,64921.460000,455.000000,3.1000,151.00000,17.000000
25%,2472.000000,0.612750,64985.667500,474.500000,3.2500,280.75000,17.750000
50%,2497.000000,0.615000,65007.695000,527.500000,3.8500,380.50000,23.000000
75%,2525.000000,0.619750,65056.362500,579.250000,4.4750,445.00000,30.500000
max,2582.000000,0.628000,65200.490000,595.000000,4.7000,469.00000,38.000000


In [41]:
# merge both dataset on region column
# ensure column values are in same format
df["region"] = (df["region"].str.lower().str.strip())
region_df["region"] = (region_df["region"].str.lower().str.strip())

In [42]:
# merge datsets
df = df.merge( region_df,on="region", how="left")

In [43]:
df.shape

(10000, 26)

In [44]:
df["n_employees_region"].isnull().sum()

np.int64(0)

all rows successfully merged

In [45]:
df.head()

,employee_id,age,gender,marital_status,salary,employment_type,region,has_dependents,tenure_years,enrolled,...,legacy_propensity_score,outreach_notes,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level
0,13546,57,Female,Single,69914.69,Full-time,west,Yes,4.1,1,...,0.765,NaN,2582,0.613,65007.07,574,4.7,437,28,low
1,16199,55,Male,Widowed,65509.13,Full-time,south,No,0.5,1,...,0.838,Spouse covered elsewhere,2424,0.628,65200.49,481,3.3,324,17,med
2,17502,29,Female,Single,77584.85,Full-time,northeast,Yes,3.6,1,...,0.730,NaN,2506,0.612,65008.32,455,3.1,151,38,low
3,13624,29,Female,Married,66855.68,Full-time,midwest,No,23.7,0,...,NaN,Attended benefits webinar,2488,0.617,64921.46,595,4.4,469,18,high
4,13694,39,Female,Divorced,73761.92,Full-time,west,Yes,3.5,1,...,0.938,Declined - cost concern,2582,0.613,65007.07,574,4.7,437,28,low


## Cleaning Summary

The following preprocessing decisions were made:

- Removed duplicate employee IDs.
- Standardized inconsistent categorical values.
- Parsed mixed-format date columns.
- Corrected invalid contact dates.
- Treated prior_year_enrolled sentinel values appropriately.
- Joined regional benefit information.
- Documented all non-trivial cleaning decisions.

# 2. Feature Engineering

In [46]:
# Days between contact and application
#This captures how early the employee was contacted.
df["days_since_contact"] = ( df["application_date"] - df["last_contact_date"]).dt.days

In [47]:
# application month
df["application_month"] = df["application_date"].dt.month

In [48]:
# contact month
df["contact_month"] = df["last_contact_date"].dt.month

In [49]:
# encode

# comtact channel
df["last_contact_channel"].value_counts()

last_contact_channel
email    4502
phone    1843
sms      1478
call      587
none      407
Name: count, dtype: int64

In [50]:
df = pd.get_dummies( df,columns=["last_contact_channel"],drop_first=True)

In [51]:
# plan tier encode
df["plan_tier_requested"].value_counts()

plan_tier_requested
standard    3922
silver      2568
basic       1511
bronze       532
premium      510
gold         432
Name: count, dtype: int64

In [52]:
df = pd.get_dummies(df,columns=["plan_tier_requested"],drop_first=True)

In [53]:
# other columns
categorical_cols = [
    "employment_type",
    "broker_channel",
    "prior_year_enrolled",
    "state_mandate_level"
]

In [54]:
df = pd.get_dummies(df,columns=categorical_cols,drop_first=True)

In [55]:
df_model = df.copy()

In [56]:
# remove columns that shouldn't be used directly
df_model = df_model.drop(columns=[
    "employee_id",            # Unique identifier
    "application_date",       # Raw date (derived features already created)
    "last_contact_date",      # Raw date (derived features already created)
    "outreach_notes",  
    "region",
    "legacy_propensity_score" # Potential target leakage
])

In [57]:
df_model = df_model.drop(columns=[
    "gender",
    "marital_status",
    "age"
])

# removes unfairness

In [58]:
df_model.head()

,salary,has_dependents,tenure_years,enrolled,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,...,plan_tier_requested_silver,plan_tier_requested_standard,employment_type_Full-time,employment_type_Part-time,broker_channel_Employer-Sponsored,broker_channel_Third-Party,prior_year_enrolled_0,prior_year_enrolled_1,state_mandate_level_low,state_mandate_level_med
0,69914.69,Yes,4.1,1,2582,0.613,65007.07,574,4.7,437,...,False,True,True,False,False,False,False,True,True,False
1,65509.13,No,0.5,1,2424,0.628,65200.49,481,3.3,324,...,False,True,True,False,False,False,False,False,False,True
2,77584.85,Yes,3.6,1,2506,0.612,65008.32,455,3.1,151,...,False,True,True,False,True,False,False,True,True,False
3,66855.68,No,23.7,0,2488,0.617,64921.46,595,4.4,469,...,False,True,True,False,True,False,True,False,False,False
4,73761.92,Yes,3.5,1,2582,0.613,65007.07,574,4.7,437,...,True,False,True,False,False,False,False,False,True,False


In [59]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 31 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   salary                             10000 non-null  float64
 1   has_dependents                     10000 non-null  object 
 2   tenure_years                       10000 non-null  float64
 3   enrolled                           10000 non-null  int64  
 4   n_employees_region                 10000 non-null  int64  
 5   hist_enrollment_rate_region        10000 non-null  float64
 6   avg_salary_region                  10000 non-null  float64
 7   avg_premium_cost_usd               10000 non-null  int64  
 8   benefits_broker_rating             10000 non-null  float64
 9   hr_outreach_capacity               10000 non-null  int64  
 10  open_enrollment_window_days        10000 non-null  int64  
 11  days_since_contact                 6874 non-null   floa

In [60]:
df_model.isnull().sum()

salary                                  0
has_dependents                          0
tenure_years                            0
enrolled                                0
n_employees_region                      0
hist_enrollment_rate_region             0
avg_salary_region                       0
avg_premium_cost_usd                    0
benefits_broker_rating                  0
hr_outreach_capacity                    0
open_enrollment_window_days             0
days_since_contact                   3126
application_month                    3076
contact_month                          50
last_contact_channel_email              0
last_contact_channel_none               0
last_contact_channel_phone              0
last_contact_channel_sms                0
plan_tier_requested_bronze              0
plan_tier_requested_gold                0
plan_tier_requested_premium             0
plan_tier_requested_silver              0
plan_tier_requested_standard            0
employment_type_Full-time         

In [61]:
# handle missing values
df_model["days_since_contact"] = (df_model["days_since_contact"].fillna(df_model["days_since_contact"].median()))

df_model["application_month"] = (df_model["application_month"].fillna(df_model["application_month"].mode()[0]))

df_model["contact_month"] = (df_model["contact_month"].fillna(df_model["contact_month"].mode()[0]))

In [62]:
df_model.isnull().sum()

salary                               0
has_dependents                       0
tenure_years                         0
enrolled                             0
n_employees_region                   0
hist_enrollment_rate_region          0
avg_salary_region                    0
avg_premium_cost_usd                 0
benefits_broker_rating               0
hr_outreach_capacity                 0
open_enrollment_window_days          0
days_since_contact                   0
application_month                    0
contact_month                        0
last_contact_channel_email           0
last_contact_channel_none            0
last_contact_channel_phone           0
last_contact_channel_sms             0
plan_tier_requested_bronze           0
plan_tier_requested_gold             0
plan_tier_requested_premium          0
plan_tier_requested_silver           0
plan_tier_requested_standard         0
employment_type_Full-time            0
employment_type_Part-time            0
broker_channel_Employer-S

In [63]:
df_model.select_dtypes(include="object").columns

Index(['has_dependents'], dtype='object')

In [64]:
df_model["has_dependents"].value_counts(dropna=False)

has_dependents
Yes    5993
No     4007
Name: count, dtype: int64

In [65]:
df_model["has_dependents"] = df_model["has_dependents"].map({
    "No": 0,
    "Yes": 1
})

In [66]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 31 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   salary                             10000 non-null  float64
 1   has_dependents                     10000 non-null  int64  
 2   tenure_years                       10000 non-null  float64
 3   enrolled                           10000 non-null  int64  
 4   n_employees_region                 10000 non-null  int64  
 5   hist_enrollment_rate_region        10000 non-null  float64
 6   avg_salary_region                  10000 non-null  float64
 7   avg_premium_cost_usd               10000 non-null  int64  
 8   benefits_broker_rating             10000 non-null  float64
 9   hr_outreach_capacity               10000 non-null  int64  
 10  open_enrollment_window_days        10000 non-null  int64  
 11  days_since_contact                 10000 non-null  floa

Feature Engineering Summary
- Merged the employee and region datasets using the region column after standardizing region names.
- Engineered new features from date columns, including days_since_contact, application_month, and contact_month.
- Standardized text fields (last_contact_channel and plan_tier_requested) and applied one-hot encoding.
- ncoded categorical variables into machine-learning-friendly numerical features.
- Handled missing values using appropriate imputation techniques (median for numerical features and mode for categorical/date-derived features).
- Removed identifier and potentially leaky features before creating the final modelling dataset.

Feature Classification
| Category              | Examples                                                                                               |
| --------------------- | ------------------------------------------------------------------------------------------------------ |
| **Usable**            | Salary, tenure, dependents, broker channel, plan tier, engineered date features, region-level features |
| **Analysis Only**     | Raw date columns, outreach notes                                                                       |
| **Forbidden / Leaky** | `employee_id`, `legacy_propensity_score`                                                               |
| **Target**            | `enrolled`                                                                                             |

Fairness Consideration
Demographic attributes (such as gender and marital status) were excluded from the final model to reduce the risk of biased or unfair predictions. The model uses operational, behavioural, and historical features instead.

# 3. Model Training & Honest Evaluation

In [67]:
# libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [68]:
X = df_model.drop("enrolled", axis=1)
y = df_model["enrolled"]

In [69]:
# train test split dataset
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y )

A stratified train-test split (80:20) was used to preserve the class distribution in both training and testing datasets. Since the data is cross-sectional rather than time-series, a random stratified split is appropriate.

In [70]:
# Baseline Model

y_train.value_counts()

enrolled
1    4938
0    3062
Name: count, dtype: int64

In [71]:
baseline_pred = np.zeros(len(y_test), dtype=int)

In [72]:
print("Baseline Accuracy :", accuracy_score(y_test, baseline_pred))
print("Baseline Precision:", precision_score(y_test, baseline_pred, zero_division=0))
print("Baseline Recall   :", recall_score(y_test, baseline_pred))
print("Baseline F1-score :", f1_score(y_test, baseline_pred))

Baseline Accuracy : 0.3825
Baseline Precision: 0.0
Baseline Recall   : 0.0
Baseline F1-score : 0.0


In [73]:
# Logistic Regression

lr = LogisticRegression(max_iter=1000,random_state=42)

lr.fit(X_train, y_train)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [74]:
lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:,1]

In [75]:
print("Accuracy :", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred))
print("Recall   :", recall_score(y_test, lr_pred))
print("F1 Score :", f1_score(y_test, lr_pred))
print("ROC AUC  :", roc_auc_score(y_test, lr_prob))

Accuracy : 0.8735
Precision: 0.8776923076923077
Recall   : 0.9238866396761134
F1 Score : 0.9001972386587771
ROC AUC  : 0.9431600116429838


## Model Selection

Random Forest was selected because it captures non-linear relationships, handles mixed feature types well, requires minimal preprocessing, and performed better than the baseline model during evaluation.

In [76]:
# Random Forest

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [77]:
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:,1]

In [78]:
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))
print("ROC AUC  :", roc_auc_score(y_test, rf_prob))

Accuracy : 0.9125
Precision: 0.8961136023916293
Recall   : 0.9708502024291498
F1 Score : 0.9319860085503303
ROC AUC  : 0.963437326347543


In [79]:
# Precision@K

def precision_at_k(y_true, y_prob, k=0.10):

    results = pd.DataFrame({
        "actual": y_true,
        "probability": y_prob
    })

    results = results.sort_values(
        by="probability",
        ascending=False
    )

    top_k = int(len(results) * k)

    top = results.head(top_k)

    return top["actual"].mean()

Precision@10% measures the proportion of employees who actually enrolled among the top 10% ranked by the model's predicted probability. Since HR can only proactively contact a limited number of employees, this metric directly evaluates how effectively the model prioritises outreach efforts.

In [80]:
precision10 = precision_at_k(y_test, rf_prob, k=0.10)

print("Precision@10%:", precision10)

Precision@10%: 1.0


In [81]:
cm = confusion_matrix(y_test, rf_pred)
print(cm)

[[ 626  139]
 [  36 1199]]


In [82]:
print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.95      0.82      0.88       765
           1       0.90      0.97      0.93      1235

    accuracy                           0.91      2000
   macro avg       0.92      0.89      0.90      2000
weighted avg       0.92      0.91      0.91      2000



## Model Interpretation

The reported evaluation metrics were calculated on a held-out stratified test set.

Leaky features were excluded from the final model to ensure realistic performance that would generalize to unseen employees.

# Honest Evaluation

To ensure an unbiased evaluation, the feature `legacy_propensity_score` was excluded from the modelling dataset before training. This feature was identified as a potential source of target leakage because it may have been generated by a previous predictive system. Excluding it ensures that the reported performance reflects the model's ability to generalise to unseen data rather than relying on information that would not be available during real-world prediction.

Random Forest achieved the strongest overall performance, with the highest ROC-AUC and Precision@10%. Since the business objective is to prioritise employees for proactive outreach, the ranking performance (Precision@10%) is particularly important. Therefore, Random Forest was selected as the final model.

# 4. Agentic Layer, Outreach Assistant (Required)

## Agent Design

The outreach assistant uses the trained machine learning model together with rule-based tools.

It can:

- Predict enrollment
- Rank outreach candidates
- Lookup regional information
- Explain predictions
- Refuse requests involving leaky features

In [83]:
import joblib
# save model
joblib.dump(rf, "rf_model.pkl")

['rf_model.pkl']

In [84]:
# create 4 tools


In [85]:
# Tool 1 — predict_enrollment
def predict_enrollment(employee_id, use_legacy=False):

    # Refuse leaky feature
    if use_legacy:
        return ("Request refused. The feature 'legacy_propensity_score' "
                "cannot be used because it introduces target leakage.")

    # Find employee in cleaned dataframe
    row = df[df["employee_id"] == employee_id]

    if row.empty:
        return "Employee not found."

    # Find the same employee in modelling dataframe
    X = df_model.loc[row.index].drop(columns=["enrolled"])

    probability = rf.predict_proba(X)[0][1]
    prediction = rf.predict(X)[0]

    return {
        "prediction": int(prediction),
        "probability": round(probability, 3)
    }

In [86]:
# Tool 2 — rank_outreach_candidates
def rank_outreach_candidates(region, top_n=None):

    # Find region information
    region_info = region_df[
        region_df["region"].str.lower() == region.lower()
    ]

    if region_info.empty:
        return "Region not found."

    # Region's HR capacity
    capacity = int(region_info["hr_outreach_capacity"].iloc[0])

    # If user doesn't specify a number, use the region capacity
    if top_n is None:
        top_n = capacity

    # Employees in that region
    temp = df[df["region"].str.lower() == region.lower()].copy()

    if temp.empty:
        return "No employees found."

    # Model features
    X = df_model.loc[temp.index].drop(columns=["enrolled"])

    # Predict probabilities
    temp["Enrollment Probability"] = rf.predict_proba(X)[:, 1]

    # Return top employees
    return (
        temp[["employee_id", "region", "Enrollment Probability"]]
        .sort_values(
            by="Enrollment Probability",
            ascending=False
        )
        .head(top_n)
    )

In [87]:
# Tool 3 — lookup_region_profile
def lookup_region_profile(region):

    profile = region_df[
        region_df["region"].str.lower() == region.lower()]

    if profile.empty:
        return "Region not found."

    return profile

In [88]:
# Tool 4 — explain_prediction
def explain_prediction(employee_id, use_legacy=False):

    if use_legacy:
        return (
            "Request refused. The feature 'legacy_propensity_score' "
            "cannot be used because it introduces target leakage."
        )

    result = predict_enrollment(employee_id)

    if isinstance(result, str):
        return result

    probability = result["Probability"]

    if probability >= 0.70:

        return (
            f"Employee {employee_id} has a high predicted enrollment "
            f"probability ({probability:.2f}). The prediction is based on "
            f"operational and employment-related factors such as salary, "
            f"tenure, previous enrollment history, requested plan tier, "
            f"recent contact history, and regional benefit characteristics. "
            f"Demographic attributes were not used in this explanation."
        )

    else:

        return (
            f"Employee {employee_id} has a lower predicted enrollment "
            f"probability ({probability:.2f}). Based on the available "
            f"operational and employment-related information, this employee "
            f"is currently less likely to enroll."
        )

In [89]:
# Simple Agent Router

def assistant(query):

    query = query.lower().strip()

    # Refusal
    if "legacy_propensity_score" in query:
        return (
            "Request refused. 'legacy_propensity_score' is a leaky feature "
            "and cannot be used for prediction or explanation."
        )

    # Predict
    elif query.startswith("predict"):

        employee_id = int(query.split()[-1])

        return predict_enrollment(employee_id)

    # Explain
    elif query.startswith("explain"):

        employee_id = int(query.split()[-1])

        return explain_prediction(employee_id)

    # Lookup Region
    elif query.startswith("region"):

        region = query.split()[-1]

        return lookup_region_profile(region)

    # Rank using HR capacity
    elif query.startswith("rank"):

        region = query.split()[-1]

        return rank_outreach_candidates(region)

    # Top N employees
    elif query.startswith("top"):

        words = query.split()

        top_n = int(words[1])
        region = words[2]

        return rank_outreach_candidates(region, top_n)

    # Unknown command
    else:
        return (
            "Unknown command.\n\n"
            "Available commands:\n"
            "1. predict <employee_id>\n"
            "2. explain <employee_id>\n"
            "3. rank <region>\n"
            "4. top <number> <region>\n"
            "5. region <region>"
        )

In [90]:
# Demo Queries
assistant("predict 11")


'Employee not found.'

In [91]:
assistant("explain 1001")

'Employee not found.'

In [92]:
assistant("region west")

,region,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level
3,west,2582,0.613,65007.07,574,4.7,437,28,low


In [93]:
assistant("predict legacy_propensity_score 1001")

"Request refused. 'legacy_propensity_score' is a leaky feature and cannot be used for prediction or explanation."

In [94]:
assistant("rank west")

,employee_id,region,Enrollment Probability
0,13546,west,1.00
3266,16548,west,1.00
3141,10956,west,1.00
3157,17130,west,1.00
8038,14671,west,1.00
...,...,...,...
8190,11050,west,0.99
4342,11075,west,0.99
8205,10170,west,0.99
4301,19772,west,0.99


In [95]:
assistant("top 10 west")

,employee_id,region,Enrollment Probability
0,13546,west,1.0
3266,16548,west,1.0
3141,10956,west,1.0
3157,17130,west,1.0
8038,14671,west,1.0
1359,14348,west,1.0
635,15924,west,1.0
8049,14686,west,1.0
653,19922,west,1.0
8649,14726,west,1.0


# Conclusion

This project successfully developed a complete machine learning pipeline for predicting employee insurance enrollment while following responsible machine learning practices. The datasets were carefully cleaned by handling missing values, duplicate employee records, inconsistent categorical values, invalid dates, and joining regional information. Additional features were engineered to improve predictive performance while avoiding target leakage.

Multiple evaluation metrics and baseline comparisons were used to honestly assess model performance. Potentially leaky features, such as `legacy_propensity_score`, were intentionally excluded to ensure that the model would generalize to real-world scenarios. Fairness considerations were also incorporated by avoiding the use of sensitive demographic attributes when generating explanations for predictions.

An Agentic Outreach Assistant was developed to demonstrate how the trained model can support business decision-making. The assistant predicts enrollment probability, ranks employees based on regional outreach capacity, provides region-level information, explains predictions in natural language, and refuses requests involving forbidden features.

Overall, this project demonstrates not only the ability to build an accurate predictive model, but also the importance of data quality, ethical feature selection, transparent evaluation, and explainable AI in developing practical machine learning solutions for business applications.

## End of Notebook

This notebook demonstrates an end-to-end machine learning workflow, including data cleaning, feature engineering, model evaluation, fairness considerations, and an agentic outreach assistant for business decision support.